# Milestones 2: Phân tích cú pháp phân cấp & Chuẩn hoá

**Nhóm:** Nhóm 11 — Nhập môn Khoa học Dữ liệu
**Đặc tả liên quan:** `parsing.md` — *Project Milestone 2 — Hierarchical Parsing and Standardization*

---

## Giới thiệu

### Tổng quan và mục đích

Milestone 1 thu thập mã nguồn LaTeX thô của từng phiên bản (`v1`, `v2`, ...) của mỗi bài báo. **Milestone 2 chuyển dữ liệu phi cấu trúc đó thành một kho dữ liệu có cấu trúc phân cấp, đã được làm sạch, chuẩn hóa và khử trùng lặp**, làm đầu vào cho phần đối sánh tài liệu tham khảo ở Milestone 3. Đầu ra cuối cùng là ba bảng:

| Bảng | Nội dung |
| :-- | :-- |
| `elements` | Mỗi dòng là một phần tử **duy nhất** (`element_id`, `arxiv_id`, `kind`, `content`) trong mọi version; nội dung giống nhau sau cleanup chỉ chiếm một dòng |
| `hierarchy` | Mỗi dòng là một cạnh nút–cha của từng version (`arxiv_id`, `version`, `element_id`, `parent_id`) |
| `bibtex` | Các mục BibTeX thống nhất sau khử trùng lặp (`arxiv_id`, `citation_key`, `entry_type`, `title`, `authors`, `year`, `doi`, `journal`, `url`) |

Báo cáo trình bày theo đúng thứ tự các thành phần của đặc tả:

| # | Phần | Module chính | Lớp / hàm chính |
| :-: | :-- | :-- | :-- |
| 1 | Multi File Gathering | `gatherer.py` | `MultiFileGatherer` |
| 2 | Hierarchy Construction | `hierarchy_builder.py` | `HierarchyBuilder` |
| 3 | Cleanup and Normalization | `hierarchy_normalizer.py` | `HierarchyNormalizer`, `normalize_path` |
| 4 | Reference Extraction | `dedup.py` (đọc `raw_references.json` do phần 2 lưu) | `ReferenceDeduplicator.parse_bib_content`, `parse_bibitem_entries`, `load_publication_references` |
| 5 | Deduplication | `dedup.py` | `ReferenceDeduplicator`, `ElementDeduplicator`, `PublicationDeduplicator` |
| 6 | Benchmark và thống kê | `benchmark.py` | `ParserBenchmark` |
| — | Điều phối toàn bộ | `pipeline.py` | `ParsingPipeline` |

### Luồng hoạt động chính (theo `pipeline.py`)

`ParsingPipeline.process_publication(pub_dir)` chạy lần lượt các bước sau cho **mỗi publication**:

1. **`build_publication`** — với từng version: gom đa file (`MultiFileGatherer.gather`) → dựng phần tử và cạnh (`HierarchyBuilder.parse_version`) → gán ID và dựng cây (`build_version_hierarchy_outputs`) → ghi các file trung gian vào `elements/`, `compilation_manifest.json`, `raw_references.json` và `vX/hierarchy_tree.json`.
2. **`normalize_publication`** — chuẩn hóa `content` của mọi nút trong các cây `hierarchy_tree*.json` **tại chỗ** (`HierarchyNormalizer.normalize_file(tree, tree)`).
3. **`deduplicate_publication`** — `PublicationDeduplicator.deduplicate_and_save_publication`: trích xuất và khử trùng lặp tài liệu tham khảo, đổi tên `\cite{}`, khử trùng lặp toàn văn, ghi ba bảng chuẩn và cập nhật lại cây từng version.
4. **Dọn file TeX thừa** (nếu `clean_redundant_tex = True`, là mặc định): `MultiFileGatherer.clean_redundant_files(..., keep_root=False)` xóa mọi file `.tex` của version sau khi đã gom.
5. **Ghi benchmark** (`ParserBenchmark.record_version` / `record_publication`) để thống kê thời gian, bộ nhớ, tỉ lệ thành công và hiệu quả khử trùng lặp (phần 6).

`run_all(limit)` lặp bước trên cho mọi publication tìm được bởi `find_publications`; hàm `main` bổ sung các tùy chọn `--source-dir`, `--clean-tex` / `--no-clean-tex`, `--report-md`, `--stats-json` và `--limit`.

### Thư viện và công cụ sử dụng

Bảng dưới đây gộp các thư viện của 6 file `pipeline.py`, `gatherer.py`, `hierarchy_builder.py`, `hierarchy_normalizer.py`, `dedup.py` và `benchmark.py` (thư viện dùng ở nhiều file chỉ mô tả một lần). Phần xử lý chính chỉ dùng **thư viện chuẩn của Python**; `pandas` chỉ dùng tùy chọn để xuất CSV/Parquet, và `psutil` là thư viện ngoài duy nhất bắt buộc (cho `benchmark.py`).

| Thư viện / công cụ | Vai trò trong Milestone 2 | Dùng bởi |
| :-- | :-- | :-- |
| `re` (biểu thức chính quy) | Xóa comment; nhận diện `\input`/`\include`/`\includeonly`; tách thân bài, mục, công thức, hình/bảng, danh sách; che token khi tách câu; thống nhất ký hiệu toán và dọn lệnh LaTeX; phân tích `\bibitem`/`.bib`; nhận diện lệnh `\cite*` để đổi tên khóa | tất cả module xử lý |
| `json` | Đọc/ghi cây, danh sách phần tử, cạnh, manifest, `raw_references.json`, bảng `bibtex` và thống kê | `pipeline`, `hierarchy_normalizer`, `dedup` |
| `pathlib.Path` | Duyệt thư mục (`glob`, `rglob`, `iterdir`), phân giải đường dẫn include, đọc/ghi file đa nền tảng | tất cả |
| `os` | `os.walk` (bottom-up) để xóa các thư mục rỗng sau khi dọn file; `os.getpid()` để lấy tiến trình hiện tại | `gatherer`, `benchmark` |
| `typing`, `from __future__ import annotations` | Chú thích kiểu (`Dict`, `List`, `Optional`, `Set`, `Tuple`, `Any`) | tất cả |
| `argparse` | Giao diện dòng lệnh: `--source-dir`, `--clean-tex`, `--limit`, ... (pipeline); `--input`, `--output-dir`, `--overwrite` (normalizer) | `pipeline`, `hierarchy_normalizer` |
| `logging` | Ghi cảnh báo khi không đọc được file JSON và log tiến độ từng publication | `pipeline` |
| `time` (`perf_counter`) | Đo thời gian xử lý từng version, từng publication và tổng thời gian chạy | `pipeline`, `benchmark` |
| `psutil` | Đo RAM vật lý (RSS) của tiến trình, ghi nhận đỉnh | `benchmark` |
| `tracemalloc` | Theo dõi bộ nhớ heap Python được cấp phát, lấy đỉnh | `benchmark` |
| `pandas` *(tùy chọn)* | Xuất `elements`, `hierarchy`, `bibtex` ra `.csv` và `.parquet` (trong khối `try/except`, thiếu thư viện thì bỏ qua) | `dedup` |
| Engine Parquet *(tùy chọn)* | Ghi file `.parquet` (lỗi được bỏ qua) | `dedup` |
| Union-Find (cài đặt tay) + bảng băm | Gom cụm các mục tài liệu tham khảo trùng lặp với chi phí gần tuyến tính | `dedup` |
| Module nội bộ | `.gatherer`, `.hierarchy_builder`, `.hierarchy_normalizer`, `.dedup`, `.benchmark` (`ParserBenchmark`) | `pipeline`; `hierarchy_builder` import `gatherer`; `dedup` import `hierarchy_builder`, `hierarchy_normalizer` |

### Sơ đồ luồng dữ liệu toàn pipeline

```text
<source_dir>/<arxiv_id>/vX/*.tex, *.bib          (kết quả của Milestone 1)
          │
          ▼
┌──────────────── 1. MULTI FILE GATHERING (gatherer.py) ────────────────┐
│ find_root_file  → chọn file gốc có \documentclass                      │
│ inline_recursive → đi theo \input / \include (có \includeonly,        │
│                    phát hiện chu trình bằng call-stack)                │
│ Đầu ra: inlined_content, compilation_files, orphan_files, cờ hoàn chỉnh│
└────────────────────────────────────────────────────────────────────────┘
          │  elements/full_inlined_vX.tex, compilation_manifest.json
          ▼
┌──────────── 2. HIERARCHY CONSTRUCTION (hierarchy_builder.py) ─────────┐
│ tách thân bài, giữ riêng References → raw_references.json              │
│ tách khối: heading / công thức / hình-bảng / danh sách / văn xuôi      │
│ tách câu an toàn (masking) → dựng cây bằng ngăn xếp tiêu đề            │
│ Đầu ra: elements_vX, hierarchy_vX, hierarchy_tree_vX (nội dung THÔ)    │
└────────────────────────────────────────────────────────────────────────┘
          │  vX/hierarchy_tree.json, elements/hierarchy_tree_vX.json
          ▼
┌──────── 3. CLEANUP AND NORMALIZATION (hierarchy_normalizer.py) ────────┐
│ chuẩn hóa toán inline/block, bỏ lệnh trình bày, giữ nguyên cấu trúc/ID │
│ Đầu ra: cây cùng cấu trúc, `content` đã chuẩn hóa (ghi tại chỗ)        │
└────────────────────────────────────────────────────────────────────────┘
          │
          ▼
┌──────────────── 4. REFERENCE EXTRACTION (dedup.py) ────────────────────┐
│ .bib trong vX/  +  \bibitem từ raw_references.json  →  mục BibTeX chuẩn│
└────────────────────────────────────────────────────────────────────────┘
          │
          ▼
┌───────────────────── 5. DEDUPLICATION (dedup.py) ─────────────────────┐
│ 5a. Khử trùng lặp tham khảo (Union-Find 3 tầng) + hợp nhất trường      │
│ 5b. Đổi tên \cite{} theo key_mapping                                   │
│ 5c. Khử trùng lặp toàn văn (sau cleanup) + ánh xạ lại hierarchy        │
│ 5d. Dựng lại cây từng version với element_id dùng chung                │
└────────────────────────────────────────────────────────────────────────┘
          │
          ▼
<pub_dir>/elements/{elements,hierarchy}.{json,csv,parquet}, dedup_stats.json
<pub_dir>/bibtex/{bibtex.{json,csv,parquet}, citation_key_mapping.json}
<pub_dir>/vX/hierarchy_tree.json
```

In [ ]:
import sys, json, re, inspect, textwrap, tempfile
import pandas as pd
from pathlib import Path

# Tự động tìm thư mục chứa package `parser` (chạy từ thư mục gốc repo, reports/ hoặc src/)
def find_src_dir():
    curr = Path.cwd().resolve()
    for _ in range(5):
        for cand in (curr / "src", curr / "NMKHDL_Nhom11" / "src", curr):
            if (cand / "parser").is_dir():
                return cand
        if curr.parent == curr:
            break
        curr = curr.parent
    return Path.cwd() / "src"

src_dir = find_src_dir()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from parser.gatherer import MultiFileGatherer
from parser.hierarchy_builder import HierarchyBuilder
from parser.hierarchy_normalizer import HierarchyNormalizer, normalize_path
from parser.dedup import ReferenceDeduplicator, ElementDeduplicator, PublicationDeduplicator
from parser.pipeline import ParsingPipeline
from parser.benchmark import ParserBenchmark

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
try:
    display
except NameError:
    display = print

def show_source(*objs):
    """In mã nguồn thật của các hàm/lớp bằng inspect (luôn khớp với file .py hiện hành)."""
    for obj in objs:
        src = textwrap.dedent(inspect.getsource(obj))
        print(f"# ===== {obj.__qualname__}  ({len(src.splitlines())} dòng) =====")
        print(src)

print("Nạp thành công 6 module từ:", src_dir / "parser")

In [ ]:
# Mã nguồn điều phối của pipeline (luồng chính cho mỗi publication)
show_source(ParsingPipeline.process_publication, ParsingPipeline.normalize_publication)

---

# 1. Multi File Gathering

## 1.1. Bối cảnh và yêu cầu

Một bài báo LaTeX thường tách thành nhiều file (`sec/1_intro.tex`, `tables/dataset.tex`, ...) được nhúng bằng `\input{...}` hoặc `\include{...}`; thư mục nguồn còn lẫn file nháp, template hay thư phản hồi phản biện. Theo `parsing.md`, parser phải xác định **file gốc** (file chứa `\documentclass`), theo `\input`/`\include` một cách bắc cầu, coi lệnh nằm trong comment là văn bản thường, và bỏ qua mọi file không thuộc đường biên dịch (compilation path).

## 1.2. Luồng dữ liệu (Data workflow)

```text
<pub_dir>/vX/  (nhiều file .tex)
      │
      ▼  find_root_file
 Chọn file gốc: \documentclass / \documentstyle (sau khi bỏ comment); nếu không có thì \begin{document};
 không có cả hai → trả về None (không chọn bừa một file lẻ)
      │
      ▼  parse_includeonly (chỉ tìm trong preamble của file gốc)
      │
      ▼  inline_recursive (đệ quy theo từng dòng)
 \include{f} · \input{f} · \input f  ──► resolve_include_path ──► nối nội dung vào đúng vị trí gọi
   • \include bị \includeonly loại → chỉ còn dòng chú thích
   • file nằm trong call-stack của chính nó → ghi chu trình (cycle_errors)
   • không tìm thấy file → unresolved_includes · lỗi đọc → read_errors
      │
      ▼
 dict kết quả: success / is_complete, root_file, compilation_files, orphan_files, inlined_content, ...
      │
      ▼  (do pipeline.py thực hiện)
 elements/full_inlined_vX.tex · elements/compilation_manifest.json · (tùy chọn) xóa file TeX thừa
```

## 1.3. Lớp và các hàm chính — `MultiFileGatherer`

Lớp chỉ gồm hằng số lớp và `@classmethod` (không giữ trạng thái).

| Thành phần | Chức năng |
| :-- | :-- |
| `COMMENT_PATTERN` | Regex xóa comment `%...` đến hết dòng, giữ `\%` hợp lệ |
| `INPUT_INCLUDE_PATTERN` | Regex bắt đồng thời `\include{f}`, `\input{f}` và `\input f` (cú pháp TeX không ngoặc nhọn) |
| `STANDARD_ROOT_NAMES`, `IGNORE_KEYWORDS` | Tên file gốc phổ biến (`main.tex`, `paper.tex`, ...) và từ khóa file phụ (`response`, `rebuttal`, `referee`, `cover_letter`, ...) |
| `strip_comments(text)` | Xóa comment LaTeX |
| `parse_includeonly(preamble_text)` | Đọc `\includeonly{...}`; trả về tập tên file được phép, hoặc `None` nếu không khai báo |
| `find_root_file(version_dir, metadata)` | Chọn file gốc. Một ứng viên → chọn luôn; nhiều ứng viên → loại file phụ theo `IGNORE_KEYWORDS`, rồi chấm điểm: +50 nếu có `\begin{document}`, +30 nếu tên thuộc `STANDARD_ROOT_NAMES`, +15 cho mỗi từ khóa của `metadata["title"]` xuất hiện trong file, cộng thêm điểm cho file nằm nông và file lớn |
| `resolve_include_path(base_dir, root_dir, target)` | Tìm file được include (có/không đuôi `.tex`), trong thư mục file hiện tại rồi thư mục gốc version; cuối cùng thử so khớp **không phân biệt hoa thường** |
| `gather(version_dir, metadata)` | Hàm điều phối: tìm file gốc, đọc `\includeonly`, gom đệ quy (hàm lồng `inline_recursive`), xác định file orphan, tính tính hoàn chỉnh |
| `clean_redundant_files(version_dir, compilation_files, orphan_files, keep_root)` | Xóa file orphan (và các file đã gom), rồi xóa các thư mục rỗng; trả về danh sách file đã xóa |

## 1.4. Cơ chế hoạt động

- **`\input` không ngoặc nhọn:** regex bắt cả `\input chapter1` (token tên file đến khoảng trắng/xuống dòng).
- **`\includeonly`:** chỉ áp dụng cho `\include` (không áp dụng cho `\input`); `\include` trỏ tới file ngoài danh sách bị bỏ qua và để lại dòng `% [Excluded by \includeonly: ...]`.
- **Phát hiện chu trình bằng call-stack:** `call_stack` chỉ chứa chuỗi tổ tiên đang mở. Một file được include nhiều lần ở các vị trí khác nhau vẫn hợp lệ; chỉ khi nó nằm trong chính chuỗi gọi của nó ($A \rightarrow B \rightarrow A$) mới bị ghi vào `cycle_errors`.
- **Comment không phải include:** mỗi dòng được xóa comment trước khi tìm lệnh include, nên `% \input{old}` được coi là văn bản thường.
- **Tính hoàn chỉnh:** `is_complete` (và `success`) chỉ đúng khi không có include nào bị thiếu, không có chu trình và không có lỗi đọc file.
- **Root nghiêm ngặt:** không có `\documentclass`, `\documentstyle` hoặc `\begin{document}` thì `find_root_file` trả về `None`, `gather` báo lỗi và đưa mọi file `.tex` vào `orphan_files`.
- **Đánh dấu vị trí nhúng:** mỗi file được nhúng nằm giữa `% --- BEGIN INPUT: <tên> ---` và `% --- END INPUT: <tên> ---` (hoặc `INCLUDE`).

## 1.5. Đầu vào và đầu ra

| | Mô tả |
| :-- | :-- |
| **Đầu vào** | `version_dir` (thư mục một version) và `metadata` tùy chọn (dict có `title`, được `pipeline` đọc từ `vX/metadata.json`) |
| **Đầu ra của `gather`** | dict: `success`, `is_complete`, `root_file`, `compilation_files` (theo thứ tự gặp), `orphan_files`, `inlined_content`, `unresolved_includes`, `cycle_errors`, `read_errors`, `error` |
| **Version không có `.tex`** | `success = False`, `error` chứa `"No .tex files found"` → `pipeline` ghi nhận **`empty_version`** (không bỏ qua) |
| **Lỗi khác** (không có file gốc, include thiếu, chu trình, lỗi đọc) | `success = False` → `pipeline` ghi nhận **`failed`** |

`MultiFileGatherer` chỉ trả kết quả **trong bộ nhớ**, không ghi file nào (docstring của module nêu rõ việc lưu thuộc về `ParsingPipeline`; module cũng không còn import `json`). Trong `pipeline.process_version`, nếu gom thất bại vì các file `.tex` đã bị dọn ở lần chạy trước mà `full_inlined_vX.tex` còn tồn tại và lần chạy trước thành công, pipeline đọc lại file đã nội tuyến để chạy lại được (idempotent) và đặt cờ `reused_inlined_content`; khi đó `build_publication` **giữ nguyên** `root_file`, `compilation_files`, `orphan_files` của manifest lần trước thay vì ghi danh sách rỗng. `build_publication` ghi `elements/full_inlined_vX.tex` và `elements/compilation_manifest.json` (trạng thái, `root_file`, `compilation_files`, `orphan_files`, và `error` nếu có).

## 1.6. Mã nguồn chính

In [ ]:
show_source(MultiFileGatherer.parse_includeonly,
            MultiFileGatherer.find_root_file,
            MultiFileGatherer.gather,
            MultiFileGatherer.clean_redundant_files)

## 1.7. Minh họa trực tiếp

Thư mục tạm có file gốc `main.tex` (có `\includeonly`), hai chương cùng `\input` một file `divider.tex` (hợp lệ, **không** phải chu trình), một chương bị loại bởi `\includeonly`, một `\input` không ngoặc nhọn, một `\input` bị comment và một file orphan.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    vdir = Path(tmp).resolve() / "v1"
    (vdir / "sec").mkdir(parents=True)
    (vdir / "divider.tex").write_text("\n% [DIVIDER]\n", encoding="utf-8")
    (vdir / "sec" / "sec1.tex").write_text("Section one.\n\\input{divider}\n", encoding="utf-8")
    (vdir / "sec" / "sec2.tex").write_text("Section two.\n\\input{divider}\n", encoding="utf-8")
    (vdir / "sec" / "excluded.tex").write_text("EXCLUDED CONTENT\n", encoding="utf-8")
    (vdir / "unbraced.tex").write_text("UNBRACED INPUT CONTENT\n", encoding="utf-8")
    (vdir / "unused.tex").write_text("Not part of the compilation path.\n", encoding="utf-8")
    (vdir / "main.tex").write_text(
        "\\documentclass{article}\n\\includeonly{sec/sec1,sec/sec2}\n\\begin{document}\n"
        "\\include{sec/sec1}\n\\include{sec/sec2}\n\\include{sec/excluded}\n"
        "\\input unbraced\n% \\input{old_draft}\n\\end{document}\n", encoding="utf-8")

    res = MultiFileGatherer.gather(vdir)
    rel = lambda p: str(Path(p).relative_to(vdir))
    print("success / is_complete :", res["success"], "/", res["is_complete"])
    print("root_file             :", rel(res["root_file"]))
    print("compilation_files     :", [rel(p) for p in res["compilation_files"]])
    print("orphan_files          :", [rel(p) for p in res["orphan_files"]])
    print("unresolved / cycles   :", res["unresolved_includes"], "/", res["cycle_errors"])
    inl = res["inlined_content"]
    print("\n[DIVIDER] xuất hiện :", inl.count("[DIVIDER]"), "lần (include hợp lệ nhiều lần)")
    print("EXCLUDED bị loại    :", "EXCLUDED CONTENT" not in inl)
    print("\\input không ngoặc  :", "UNBRACED INPUT CONTENT" in inl)

    # Không có \documentclass -> không chọn bừa file lẻ
    lone = Path(tmp).resolve() / "lone"; lone.mkdir()
    (lone / "table.tex").write_text("\\begin{tabular}{c} a \\end{tabular}", encoding="utf-8")
    print("Root của thư mục chỉ có file bảng:", MultiFileGatherer.find_root_file(lone))

    # Thư mục không có file .tex -> version rỗng
    nothing = Path(tmp).resolve() / "nothing"; nothing.mkdir()
    print("Thư mục không có .tex ->", MultiFileGatherer.gather(nothing)["error"])

---

# 2. Hierarchy Construction

## 2.1. Bối cảnh và yêu cầu

Từ chuỗi LaTeX đã gom của một version, parser dựng **cây phân cấp**: nút gốc là `Document`; các `Chapter`/`Section` ở mức hai; các `Subsection`, `Paragraph`... ở các mức dưới. Nút lá là **Câu** (tách theo dấu chấm), **Công thức khối** và **Hình** (bảng cũng tính là hình). Khối `itemize` là nút nhánh, mỗi `\item` là một nút ở mức sâu hơn. Mục **References không đưa vào cây**; **Acknowledgements** và **Appendix bắt buộc đưa vào**. Version không có TeX phải được ghi nhận là bản ghi rỗng.

## 2.2. Luồng dữ liệu (Data workflow)

```text
inlined_content (LaTeX đã gom)           [pipeline: gather_res["inlined_content"]]
      │
      ▼  extract_body_and_preprocess
 • lấy phần giữa \begin{document} … \end{document}
 • tách thebibliography / \bibitem / \bibliography → preserved_refs  (loại khỏi thân bài)
 • abstract, acknowledgments → \section*{...};  appendix → \appendix hoặc \section*{Appendix}
 • bỏ \title, \author, \maketitle, ...
      │
      ▼  _extract_top_level_blocks
 chuỗi khối theo thứ tự: heading │ appendix_marker │ block_formula │ figure │ list │ prose
      │
      ▼  duyệt từng khối với ngăn xếp tiêu đề (heading_stack: (cấp, id))
 heading  → pop các mục cùng cấp/thấp hơn, cha = đỉnh ngăn xếp, push mục mới
            (mục References và mục con của nó: lưu vào preserved_refs, KHÔNG vào cây)
 prose    → split_sentences_safe → các nút `sentence`
 formula / figure → một nút lá;   list → process_list_block (itemize → item → lá)
      │
      ▼
 elements (raw_id) + hierarchy (cạnh cha–con) + preserved_refs            [parse_version]
      │
      ▼  build_version_hierarchy_outputs
 gán ID chính thức <arxiv_id>#root<v>, <arxiv_id>#v<v>_<tiền tố><n> → build_nested_tree → cây lồng nhau
```

## 2.3. Lớp và các hàm chính — `HierarchyBuilder`

| Hàm | Chức năng |
| :-- | :-- |
| `extract_body_and_preprocess(raw_tex)` | Tiền xử lý như sơ đồ trên; trả về `(body, preserved_refs)` với `preserved_refs` gồm `thebibliography_blocks`, `bibitem_entries`, `bibliography_commands`, `reference_sections` |
| `split_sentences_safe(text)` | Tách câu bằng **che token (masking)**: che toán inline (`$...$`, `\(...\)`), URL, lệnh LaTeX có tham số (`\cite{}`, `\ref{}`), số thập phân, từ viết tắt (`i.e.`, `e.g.`, `et al.`, `Fig. 1`, ...); tách tại `. ! ?`; khôi phục **100% chuỗi LaTeX gốc** |
| `_extract_top_level_blocks(text)` | Một regex gộp cắt văn bản thành các khối theo thứ tự; công thức gồm các môi trường toán, `$$...$$` và `\[...\]`; hình gồm `figure`, `table`, `algorithm`; danh sách được cắt bằng bộ đếm độ sâu `\begin/\end` nên bắt đúng danh sách lồng nhau |
| `_split_list_items(list_block)` | Tách các `\item` cùng cấp (không cắt nhầm `\item` của danh sách con) |
| `parse_version(arxiv_id, version, inlined_tex)` | Dựng nút gốc `root<v>`, dùng `HEADING_LEVELS` (`part`/`chapter`=1, `section`=2, `subsection`=3, `subsubsection`=4, `paragraph`=5, `subparagraph`=6) và ngăn xếp tiêu đề; xử lý Appendix; loại mục References (`REFERENCE_TITLES`); trả về `(elements, hierarchy, preserved_refs)` |
| `build_version_hierarchy_outputs(...)` | Gán ID chính thức, tạo bảng cạnh và cây; phân biệt `status = "success"`, `"empty_version"` và `"failed"` |
| `build_nested_tree(...)` | Dựng cây JSON lồng nhau từ elements và cạnh; kiểm tra nút gốc, cạnh hợp lệ và chu trình |
| `load_version_tex_from_elements_or_source(...)` | Đọc `elements/full_inlined_vX.tex`; nếu chưa có thì gọi `MultiFileGatherer.gather` |

## 2.4. Cơ chế hoạt động

- **Nút gốc chỉ làm cha:** `<arxiv_id>#root<v>` có trong `elements` nhưng không sinh cạnh có `parent_id = null`; mọi cạnh đều có `parent_id` khác null.
- **Raw Content:** các nút lá giữ nguyên LaTeX thô (kể cả `\centering`, `[htpb]`, `\midrule`); việc làm sạch thuộc phần 3.
- **Mã định danh theo ba lớp:** (1) `raw_id` tạm thời trong `parse_version` (`s_1`, `formula_2`, ...); (2) ID theo version do `build_version_hierarchy_outputs` gán; (3) ID dùng chung cho cả publication do phần Deduplication gán (mục 5.5).

| `kind` | Tiền tố ID theo version | Tiền tố ID cuối cùng (sau dedup) |
| :-- | :-: | :-: |
| `document` | `root<v>` | `root<v>` |
| `sentence` | `s` | `s` |
| `part`, `chapter`, `section` | `h` | `sec` |
| `subsection` | `ssec` | `ssec` |
| `subsubsection`, `paragraph`, `subparagraph` | `par` | `par` |
| `block_formula` | `eq` | `eq` |
| `figure` | `fig` | `fig` |
| `itemize` | `list` | `list` |
| `item` | `item` | `item` |

- **Nội dung lặp có chủ đích:** nút `itemize` chứa nguyên khối danh sách, còn `item` và các nút lá bên dưới chứa từng phần của nội dung đó.

## 2.5. Đầu vào và đầu ra

| | Mô tả |
| :-- | :-- |
| **Đầu vào** | `arxiv_id`, tên version (`"v1"`), chuỗi LaTeX đã gom |
| **`parse_version`** | `elements` (`raw_id`, `kind`, `content`), `hierarchy` (`arxiv_id`, `version`, `element_raw_id`, `parent_raw_id`), `preserved_refs` |
| **`build_version_hierarchy_outputs`** | dict: `status`, `is_empty`, `error`, `elements` (`element_id`, `arxiv_id`, `kind`, `content`), `hierarchy` (`arxiv_id`, `version`, `element_id`, `parent_id`), `tree` |
| **Cây (`tree`)** | `arxiv_id`, `version`, `version_name`, `status`, `is_empty`, `elements_count`, `edges_count`, `root` (nút: `element_id`, `kind`, `content`, `children`); version rỗng hoặc lỗi → `root = null` |
| **File do `pipeline.build_publication` ghi** | `vX/hierarchy_tree.json`; `elements/{hierarchy_tree_vX, elements_vX, hierarchy_vX, pre_dedup_elements_vX, pre_dedup_hierarchy_vX}.json`; `elements/raw_references.json` (theo version) |

## 2.6. Mã nguồn chính

In [ ]:
show_source(HierarchyBuilder.extract_body_and_preprocess,
            HierarchyBuilder.split_sentences_safe,
            HierarchyBuilder._extract_top_level_blocks,
            HierarchyBuilder._split_list_items)

In [ ]:
show_source(HierarchyBuilder.parse_version,
            HierarchyBuilder.build_version_hierarchy_outputs,
            HierarchyBuilder.build_nested_tree)

## 2.7. Minh họa trực tiếp

Đoạn LaTeX mẫu có abstract, section, câu chứa `\cite`, `i.e.`, số thập phân và `Fig. 1`, công thức `align*` và `$$...$$`, danh sách, hình có `[htpb]`, Acknowledgements, Appendix và mục References.

In [ ]:
SAMPLE_TEX = r"""
\documentclass{article}
\begin{document}
\title{Demo}
\maketitle
\begin{abstract}
We study citation matching, i.e. linking references to papers.
\end{abstract}
\section{Introduction}
Citation matching is hard \cite{vaswani2017}. The accuracy reaches 93.5 percent, see Fig. 1.
\begin{align*}
  a &= b + c \\
  d &= e
\end{align*}
Energy is given by $$ E = mc^2 $$ in this case.
\subsection{Key points}
\begin{itemize}
\item First point.
\item Second point with $x^2$. It has two sentences.
\end{itemize}
\begin{figure}[htpb]
\centering
\includegraphics[width=\linewidth]{fig1.png}
\caption{Overview of the model.}
\end{figure}
\begin{acknowledgments}
We thank the reviewers.
\end{acknowledgments}
\appendix
\section{Extra proofs}
Proof is omitted.
\section{References}
\begin{thebibliography}{9}
\bibitem{vaswani2017} A. Vaswani, N. Shazeer. ``Attention is all you need.'' NeurIPS, 2017.
\end{thebibliography}
\end{document}
"""

ARXIV_ID = "2510.10788"
raw_elements, raw_hierarchy, preserved_refs = HierarchyBuilder.parse_version(ARXIV_ID, "v1", SAMPLE_TEX)
built = HierarchyBuilder.build_version_hierarchy_outputs(ARXIV_ID, "v1", raw_elements, raw_hierarchy)
tree = built["tree"]

print(f"status = {built['status']} | {len(built['elements'])} elements | {len(built['hierarchy'])} cạnh")
print("\nBẢNG 1: ELEMENTS (ID theo version, nội dung còn thô)")
display(pd.DataFrame(built["elements"])[["element_id", "kind", "content"]])
print("\nBẢNG 2: CẠNH CHA–CON")
display(pd.DataFrame(built["hierarchy"])[["version", "element_id", "parent_id"]])

print("\nNhững gì được GIỮ RIÊNG khỏi cây (preserved_refs):")
for k, v in preserved_refs.items():
    print(f"  {k:24s}: {len(v)} mục")

empty = HierarchyBuilder.build_version_hierarchy_outputs(ARXIV_ID, "v2", [], [], is_empty=True)
failed = HierarchyBuilder.build_version_hierarchy_outputs(ARXIV_ID, "v3", [], [], status="failed", error_msg="demo")
print("\nVersion rỗng :", empty["status"], "| root =", empty["tree"]["root"])
print("Version lỗi  :", failed["status"], "| root =", failed["tree"]["root"])

---

# 3. Cleanup and Normalization

## 3.1. Bối cảnh và yêu cầu

Theo `parsing.md`, dữ liệu trong cây phải được làm sạch và chuẩn hóa:

- **LaTeX Cleanup:** bỏ các lệnh định dạng không mang ngữ nghĩa như `\centering`, tùy chọn `[htpb]`, `\midrule`.
- **Math Normalization:** đưa toán trong dòng về dạng `$...$` và toán khối về dạng `\begin{equation}...\end{equation}`.
- Việc so khớp toàn văn ở phần 5 phải thực hiện **sau** cleanup, để khác biệt nhỏ về định dạng hoặc khoảng trắng không làm hai nội dung giống nhau bị coi là khác nhau.

## 3.2. Luồng dữ liệu (Data workflow)

```text
<pub_dir>/vX/hierarchy_tree.json  và  <pub_dir>/elements/hierarchy_tree_vX.json      (cây nội dung THÔ từ phần 2)
      │   ParsingPipeline.normalize_publication  →  HierarchyNormalizer.normalize_file(tree, tree)   (ghi tại chỗ)
      ▼
 normalize_path / _normalized_file_content          (đọc JSON UTF-8; lỗi đọc hoặc JSON sai → ValueError)
      ▼
 normalize_tree: kiểm tra cây, sao chép _copy_json_tree (lặp, không đệ quy), chuẩn hóa từng nút _normalize_node
      ▼  với mỗi nút:  normalize_content(content, kind)
        1) standardize_math      : \( … \) → $ … $ ;   $$ … $$ và \[ … \] → \begin{equation} … \end{equation}
        2) nếu kind == "block_formula": _normalize_block_math_environments
                                        (equation/align/gather/multline/eqnarray/flalign/displaymath → equation)
        3) clean_latex           : bỏ comment, bỏ CLEANUP_COMMANDS, bỏ [htpb] của figure/table, gọn khoảng trắng
      ▼
 json.dumps(ensure_ascii=False, indent=2)  →  cây cùng cấu trúc và cùng ID, chỉ `content` thay đổi
```

## 3.3. Lớp và các hàm chính — `HierarchyNormalizer` và hàm cấp module

Lớp chỉ chứa hằng số lớp, `@classmethod` và một `@staticmethod` (không giữ trạng thái).

| Thành phần | Chức năng |
| :-- | :-- |
| `CLEANUP_COMMANDS` | Danh sách regex lệnh trình bày cần bỏ: `\centering`, `\raggedright`, `\raggedleft`, `\small`, `\footnotesize`, `\large`, `\Large`, `\normalsize`, `\midrule`, `\toprule`, `\bottomrule`, `\hline`, `\clearpage`, `\newpage`, `\noindent`, `\label{...}`, `\vspace{...}`, `\hspace{...}`, `\smallskip`, `\medskip`, `\bigskip` |
| `BLOCK_MATH_ENVIRONMENTS`, `BLOCK_MATH_PATTERN` | Họ môi trường toán khối (có/không dấu `*`) và regex khớp **từng thẻ** `\begin{env}` hoặc `\end{env}` với nhóm có tên `action` và `env` |
| `standardize_math(text)` | Thống nhất ký hiệu toán trong dòng và toán khối dạng `$$`/`\[` |
| `clean_latex(text)` | Xóa comment, các lệnh trong `CLEANUP_COMMANDS`, tùy chọn vị trí của `figure`/`table`; gộp khoảng trắng và dòng trống |
| `normalize_content(content, kind)` | Chuẩn hóa nội dung **một** nút theo ba bước ở sơ đồ; bước 2 chỉ áp dụng cho `kind == "block_formula"` |
| `_normalize_block_math_environments(content)` | Ghép cặp thẻ bằng **ngăn xếp**, thay môi trường toán khối được ghép cặp chắc chắn bằng `\begin{equation}` + thân + `\end{equation}` |
| `normalize_tree(tree)` | Kiểm tra (`dict`, có khóa `root`), sao chép cây, chuẩn hóa mọi nút; `root = null` được chấp nhận |
| `_copy_json_tree(value)` | Sao chép cấu trúc JSON bằng ngăn xếp tường minh (thay `deepcopy`, tránh giới hạn đệ quy) |
| `_normalize_node(node, active_nodes)` | Duyệt cây theo chiều sâu **không đệ quy**, kiểm tra kiểu từng nút/`content`/`children`, phát hiện chu trình bằng tập `id` các nút đang trên đường đi |
| `normalize_file(input_path, output_path)` | Chuẩn hóa một file; `output_path` được phép trùng `input_path` vì nội dung được tuần tự hóa xong rồi mới ghi |
| `_normalized_file_content(input_path)` | Đọc, chuẩn hóa và trả về chuỗi JSON; **không ghi** ra đĩa |
| `normalize_path(input_path, output_dir, overwrite)` | Chuẩn hóa một file hoặc mọi `hierarchy_tree*.json` trong thư mục (xem 3.4) |
| `main(argv)` | CLI `python -m parser.hierarchy_normalizer --input ... [--output-dir ...] [--overwrite]` |

## 3.4. Cơ chế hoạt động

**Ghép cặp thẻ toán khối bằng ngăn xếp.** Duyệt các thẻ theo thứ tự xuất hiện: thẻ `begin` đánh dấu mọi khung đang mở là "chứa môi trường lồng" rồi đẩy khung mới; thẻ `end` chỉ ghép khi khung trên cùng cùng loại. Các thay thế được áp dụng từ cuối chuỗi về đầu để không lệch vị trí.

| Tình huống trong `content` | Kết quả |
| :-- | :-- |
| `\begin{align*}…\end{align*}` đơn lẻ | Thay bằng `\begin{equation}\n<thân>\n\end{equation}` |
| Nhiều môi trường liên tiếp, không lồng nhau | Mỗi môi trường được thay riêng |
| Môi trường lồng nhau | **Giữ nguyên** cả môi trường ngoài lẫn trong |
| Thẻ đóng khác loại thẻ mở gần nhất | Thẻ đóng bị bỏ qua, không ghép sai |
| Thẻ mở không có thẻ đóng | Không thay; các môi trường đứng sau nó trong cùng nút cũng bị coi là lồng nhau và giữ nguyên |

**`normalize_path`.**

| | Đầu vào là **thư mục** | Đầu vào là **một file** |
| :-- | :-- | :-- |
| File xử lý | `sorted(rglob("hierarchy_tree*.json"))`, loại file có `stem` kết thúc `.normalized`; rỗng → `FileNotFoundError` | Chính file đó; đuôi phải là `.json`; **không được** kết thúc `.normalized` (`ValueError`) |
| Thư mục đầu ra mặc định | `<cha>/<tên input>_normalized` | Cùng thư mục với file đầu vào |
| `--output-dir` | Không được nằm trong thư mục đầu vào (`ValueError`) | Dùng thư mục đó |
| File đầu ra | `<stem>.normalized.json`, giữ cấu trúc thư mục `<arxiv_id>/vX/` | `<stem>.normalized.json` |

- **Hai pha:** pha chuẩn bị tính đường dẫn, kiểm tra `FileExistsError` (khi chưa có `overwrite`) và chuẩn hóa **toàn bộ** nội dung trong bộ nhớ; chỉ khi mọi file thành công mới sang pha ghi. Một file lỗi thì không file nào được ghi.
- **Trong pipeline:** `ParsingPipeline.normalize_publication` không dùng `normalize_path` mà gọi `HierarchyNormalizer.normalize_file(tree_file, tree_file)` cho cả `vX/hierarchy_tree.json` và `elements/hierarchy_tree_vX.json`.

## 3.5. Đầu vào và đầu ra

| Hàm | Đầu vào | Đầu ra |
| :-- | :-- | :-- |
| `normalize_content(content, kind)` | Chuỗi nội dung và `kind` của nút | Chuỗi đã chuẩn hóa toán và làm sạch |
| `normalize_tree(tree)` | `dict` cây có khóa `root` | **Bản sao** với `content` đã chuẩn hóa; cấu trúc, `element_id`, `kind` và các trường khác giữ nguyên |
| `normalize_file(input_path, output_path)` | Đường dẫn file JSON và đường dẫn đầu ra | File JSON UTF-8, `indent=2`, kết thúc bằng xuống dòng; trả về `output_path` |
| `normalize_path(...)` | File hoặc thư mục chứa `hierarchy_tree*.json` | Danh sách đường dẫn `*.normalized.json` |

**Lỗi được kiểm soát:** `FileNotFoundError` (đường dẫn không tồn tại, thư mục không có cây), `ValueError` (không đọc được/JSON sai, cây không phải object hoặc thiếu `root`, nút không phải object, `content` không phải chuỗi, `children` không phải danh sách, có chu trình, đầu vào không phải `.json` hoặc đã `.normalized`, thư mục đầu ra nằm trong đầu vào), `FileExistsError` (file đầu ra đã có mà không có `--overwrite`).

## 3.6. Mã nguồn chính

In [ ]:
show_source(HierarchyNormalizer.standardize_math,
            HierarchyNormalizer.clean_latex,
            HierarchyNormalizer.normalize_content,
            HierarchyNormalizer._normalize_block_math_environments)

In [ ]:
show_source(HierarchyNormalizer.normalize_tree,
            HierarchyNormalizer._copy_json_tree,
            HierarchyNormalizer._normalize_node,
            HierarchyNormalizer.normalize_file,
            HierarchyNormalizer._normalized_file_content,
            normalize_path)

## 3.7. Minh họa trực tiếp

**(a) `normalize_content`** trên các loại nút, gồm các trường hợp ghép cặp thẻ đặc biệt.

In [ ]:
samples = [
    ("sentence",      r"Let \(x\) be a real variable and $y$ a parameter."),
    ("block_formula", "\\begin{align*}\n  a &= b + c \\\\\n  d &= e\n\\end{align*}"),
    ("block_formula", r"$$ x^2 + y^2 = z^2 $$"),
    ("block_formula", r"\begin{equation} a \end{equation} \begin{gather} b \end{gather}"),
    ("block_formula", r"\begin{equation} \begin{align} a &= b \end{align} \end{equation}"),
    ("block_formula", r"\begin{align} x = 1 \end{equation}"),
    ("block_formula", r"\begin{align} x = 1"),
    ("figure",        r"\begin{figure}[htpb] \centering \includegraphics{a.png} \caption{Overview} \label{fig:a} \end{figure}"),
    ("sentence",      "We   study   the   problem.   % a comment"),
]
notes = ["toán inline", "align* → equation", "$$…$$ → equation", "hai môi trường liên tiếp: thay riêng",
         "lồng nhau: giữ nguyên", "thẻ đóng sai loại: giữ nguyên", "thẻ mở không đóng: giữ nguyên",
         "figure: bỏ [htpb], \\centering, \\label", "bỏ comment, gọn khoảng trắng"]
rows = [{"kind": k, "Ghi chú": n, "Trước": c, "Sau": HierarchyNormalizer.normalize_content(c, kind=k)}
        for (k, c), n in zip(samples, notes)]
display(pd.DataFrame(rows))

**(b) `normalize_tree`** trên cây vừa dựng ở mục 2.7: chỉ `content` thay đổi; cấu trúc, ID, `kind` giữ nguyên; cây gốc không bị sửa. Ô thứ hai chạy trên cây sâu 5000 mức, và ô thứ ba chạy `normalize_path` với một thư mục có file lỗi để minh họa chế độ hai pha.

In [ ]:
def flatten(node, parent=None):
    out, stack = [], [(node, parent)]
    while stack:
        cur, par = stack.pop()
        out.append({"element_id": cur["element_id"], "kind": cur["kind"], "parent_id": par,
                    "n_children": len(cur.get("children") or []), "content": cur.get("content")})
        stack.extend((c, cur["element_id"]) for c in reversed(cur.get("children") or []))
    return out

snapshot = json.dumps(tree, sort_keys=True)
normalized_tree = HierarchyNormalizer.normalize_tree(tree)
before, after = pd.DataFrame(flatten(tree["root"])), pd.DataFrame(flatten(normalized_tree["root"]))
shape = ["element_id", "kind", "parent_id", "n_children"]
cmp_df = before[shape].copy(); cmp_df["content_trước"] = before["content"]; cmp_df["content_sau"] = after["content"]
display(cmp_df[before["content"] != after["content"]])
print(f"Số nút: {len(before)} → {len(after)} | số nút đổi content: {int((before['content'] != after['content']).sum())}")
print("Cấu trúc, ID, kind giữ nguyên:", before[shape].equals(after[shape]))
print("Cây đầu vào không bị sửa     :", json.dumps(tree, sort_keys=True) == snapshot)
print("Version rỗng (root = None)   :", HierarchyNormalizer.normalize_tree(empty["tree"])["root"])

In [ ]:
# Cây rất sâu: duyệt/sao chép bằng ngăn xếp tường minh nên không chạm giới hạn đệ quy
DEPTH = 5000
deep = {"root": {"element_id": "n0", "kind": "section", "content": r"Level 0 with \(x\)", "children": []}}
node = deep["root"]
for i in range(1, DEPTH):
    child = {"element_id": f"n{i}", "kind": "sentence", "content": f"Level   {i}", "children": []}
    node["children"].append(child); node = child
res_deep = HierarchyNormalizer.normalize_tree(deep)
d, node = 1, res_deep["root"]
while node["children"]:
    node = node["children"][0]; d += 1
print(f"Giới hạn đệ quy: {sys.getrecursionlimit()} | chuẩn hóa thành công cây sâu {d} mức, nút cuối: {node['content']!r}")

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    ok_root = tmp / "sourcesDemo"
    for v in ("v1", "v2"):
        (ok_root / ARXIV_ID / v).mkdir(parents=True)
        (ok_root / ARXIV_ID / v / "hierarchy_tree.json").write_text(json.dumps(tree), encoding="utf-8")
    outs = normalize_path(ok_root)
    print("\nĐầu ra mặc định:", [str(p.relative_to(tmp)) for p in outs])
    try:
        normalize_path(ok_root)
    except FileExistsError as e:
        print("Chạy lại không overwrite -> FileExistsError")
    print("Với overwrite=True ghi lại", len(normalize_path(ok_root, overwrite=True)), "file")

    bad_root = tmp / "sourcesBad"
    for name, text in (("a", json.dumps(tree)), ("b", "{ not valid json")):
        (bad_root / name).mkdir(parents=True); (bad_root / name / "hierarchy_tree.json").write_text(text, encoding="utf-8")
    try:
        normalize_path(bad_root)
    except ValueError as e:
        print("\nFile hỏng -> ValueError; số file đã ghi:", len(list(tmp.glob("sourcesBad_normalized/**/*.json"))))
    try:
        normalize_path(outs[0])
    except ValueError as e:
        print("Đầu vào đã chuẩn hóa -> ValueError:", str(e).replace(str(tmp), "<tmp>"))

---

# 4. Reference Extraction

## 4.1. Bối cảnh và yêu cầu

Với các tài liệu tham khảo khai báo bằng `\bibitem` trong file `.tex`, parser phải chuyển chúng thành mục BibTeX chuẩn bằng công cụ lập trình như biểu thức chính quy. Mã nguồn hiện có hai nơi chứa tham khảo: file `.bib` trong thư mục version và các `\bibitem` bên trong môi trường `thebibliography`.

## 4.2. Luồng dữ liệu (Data workflow)

```text
Phần 2: extract_body_and_preprocess  ──►  raw_references.json   {vX: {status, thebibliography_blocks, bibitem_entries,
      │                                                          bibliography_commands, reference_sections}}
      ▼
ReferenceDeduplicator.load_publication_references(pub_dir, version_names)
   1. <pub_dir>/bibtex/  (nếu đã có: bibtex_vX.json / references_vX.json / vX.json; raw_bibtex.json, bibtex_raw.json,
                           all_references.json) → standardize_entry_dict
   2. với mỗi version CHƯA nạp ở bước 1:
        • mọi file **/*.bib trong vX/  (bỏ file mẫu sample-base / ws-pro-sample / acmart-sample nếu có file thật)
              → parse_bib_content
        • bibitem_entries từ elements/raw_references.json; nếu rỗng, quét thebibliography/references trong
          full_inlined_vX.tex (hoặc các .tex của version)  → parse_bibitem_entries
   3. nếu vẫn rỗng: đọc *.bib hoặc bibtex.json trong <pub_dir>/bibtex/
      ▼
 danh sách mục chuẩn {arxiv_id, version, citation_key, entry_type, title, authors, year, doi, journal, url}
```

## 4.3. Các hàm chính (trong `ReferenceDeduplicator`)

| Hàm | Chức năng |
| :-- | :-- |
| `strip_latex_accents_and_braces(text)` | Bỏ dấu LaTeX (`\"{a}` → `a`), lệnh định dạng chữ (`\emph`, `\textit`, ...) và ngoặc nhọn |
| `normalize_doi(doi)` | Về chữ thường, bỏ tiền tố `doi.org`/`doi:`; chỉ chấp nhận cấu trúc `10.xxxx/...`, ngược lại trả về chuỗi rỗng |
| `clean_display_title(title)` | Làm sạch tiêu đề để lưu vào bảng (bỏ dấu chấm cuối thừa) |
| `parse_authors_list(raw)` | Tách tên tác giả theo `and` hoặc `;`; ngoài ra tách theo dấu phẩy **chỉ khi tác giả kế tiếp viết dạng chữ viết tắt** (`A. Vaswani, N. Shazeer` → hai tác giả) để không cắt nhầm dạng BibTeX `Họ, Tên`; bỏ `others`/`et al.` |
| `parse_year_int(raw)` | Lấy năm 4 chữ số (1800–2035) hoặc `None` |
| `_extract_balanced_value(text, start)` | Đọc giá trị trường `.bib` dạng `{...}` (đếm độ sâu ngoặc nhọn), `"..."` hoặc giá trị trần |
| `parse_bib_content(bib_text, arxiv_id, version)` | Phân tích `.bib`: bỏ comment, tìm tiêu đề `@type{key,`, tìm ngoặc đóng bằng bộ đếm độ sâu, bỏ `@string/@preamble/@comment`, đọc các trường; `journal` lấy từ `journal`/`journaltitle`/`booktitle`/`series`/`publisher`; nếu thiếu `url` mà có `eprint` dạng arXiv thì tạo URL `arxiv.org/abs/...` |
| `parse_bibitem_entries(bibitems, arxiv_id, version)` | Chuyển `\bibitem[...]{key} nội dung` thành mục BibTeX bằng regex (xem 4.4) |
| `standardize_entry_dict(raw, arxiv_id, default_version)` | Đưa một dict mục bất kỳ (nhiều tên khóa khác nhau) về schema thống nhất |
| `load_publication_references(pub_dir, version_names)` | Điều phối thu thập từ mọi nguồn như sơ đồ trên |

## 4.4. Cơ chế trích xuất `\bibitem`

`parse_bibitem_entries` lấy `citation_key` và phần nội dung bằng regex, xóa comment, gộp khoảng trắng, tìm `url` (`\url{}` hoặc `http...`), `doi` (`doi:` hoặc `doi.org/10.xxxx/...`) và `year`; sau đó tách **tác giả / tiêu đề / nơi xuất bản** theo thứ tự ưu tiên:

1. Có `\newblock`: khối 1 = tác giả, khối 2 = tiêu đề, các khối còn lại = nơi xuất bản.
2. Có chuỗi trong ``...'' hoặc "...": đó là tiêu đề; phần trước là tác giả, phần sau là nơi xuất bản.
3. Có phần in nghiêng (`\emph{}`, `\textit{}`, `{\em ...}`): phần in nghiêng là tiêu đề.
4. Dạng thiên văn/vật lý `Tác giả 2023, Tạp chí, Tập, Trang`.
5. Còn lại: tách theo dấu chấm nhưng không tách sau chữ viết tắt tên (`A. B.`); nếu phần cuối của nơi xuất bản chỉ là năm trùng với `year` thì bỏ năm đó khỏi `journal`.

`entry_type` được đoán từ nội dung: `inproceedings` (proceedings/conference/symposium/workshop), `book` (press/springer/wiley/edition/book), `phdthesis` (thesis), `article` (có nơi xuất bản), ngược lại `misc`.

## 4.5. Đầu vào và đầu ra

| | Mô tả |
| :-- | :-- |
| **Đầu vào** | `pub_dir`, danh sách tên version; các `.bib` trong `vX/`, `elements/raw_references.json`, (tùy chọn) dữ liệu JSON/`.bib` trong `bibtex/` |
| **Đầu ra** | `List[dict]` các mục thô theo version với 10 trường: `arxiv_id`, `version`, `citation_key`, `entry_type`, `title`, `authors` (danh sách), `year` (int hoặc `None`), `doi`, `journal`, `url` — là đầu vào của phần 5.1 |

## 4.6. Mã nguồn chính

In [ ]:
show_source(ReferenceDeduplicator.normalize_doi,
            ReferenceDeduplicator.parse_authors_list,
            ReferenceDeduplicator._extract_balanced_value,
            ReferenceDeduplicator.parse_bib_content,
            ReferenceDeduplicator.parse_bibitem_entries)

In [ ]:
show_source(ReferenceDeduplicator.load_publication_references)

## 4.7. Minh họa trực tiếp

Một file `.bib` có ngoặc nhọn lồng nhau và dấu LaTeX, cùng các `\bibitem` ở bốn kiểu định dạng khác nhau (trong đó có mục lấy từ `preserved_refs` của mục 2.7).

In [ ]:
BIB = r"""
@inproceedings{vit_transformer,
  title  = {{Attention Is All You Need}},
  author = {Vaswani, Ashish and Shazeer, Noam and M{\"u}ller, Hans},
  booktitle = {Advances in Neural Information Processing Systems},
  year   = {2017},
  doi    = {https://doi.org/10.5555/3295222.3295349}
}
@article{kabashin2009,
  title   = "Phase-sensitive plasmonic biosensor",
  author  = {Kabashin, A. V. and Patskovsky, S.},
  journal = {Nature Materials}, year = 2009, eprint = {0901.12345}
}
@string{nm = "Nature Materials"}
"""
bib_rows = ReferenceDeduplicator.parse_bib_content(BIB, ARXIV_ID, "v2")
print("(a) parse_bib_content — .bib có ngoặc nhọn lồng, @string bị bỏ qua")
display(pd.DataFrame(bib_rows).drop(columns=["arxiv_id"]))

BIBITEMS = [
    preserved_refs["bibitem_entries"][0],   # từ mục 2.7 (dạng ``tiêu đề'')
    r"\bibitem{lecun15} Y. LeCun, Y. Bengio, G. Hinton. \newblock Deep learning. \newblock Nature, 521:436--444, 2015. doi:10.1038/nature14539",
    r"\bibitem[Ri(2014)]{Ri14} S. Riehl, \emph{Categories in context}, Dover Press, 2014.",
    r"\bibitem{astro} Smith J., Jones K. 2021, ApJ, 900, 12, Spectral survey of nearby galaxies in the optical band",
]
print("(b) parse_bibitem_entries — bốn kiểu định dạng \\bibitem")
bibitem_rows = ReferenceDeduplicator.parse_bibitem_entries(BIBITEMS, ARXIV_ID, "v1")
display(pd.DataFrame(bibitem_rows).drop(columns=["arxiv_id"]))
print("authors của lecun15 (repr):", bibitem_rows[1]["authors"])
print("dấu phẩy + chữ viết tắt :", ReferenceDeduplicator.parse_authors_list("A. Vaswani, N. Shazeer"))
print("dạng BibTeX Họ, Tên    :", ReferenceDeduplicator.parse_authors_list("Vaswani, Ashish"))

---

# 5. Deduplication

## 5.1. Bối cảnh và yêu cầu

Khử trùng lặp ở hai cấp, **xuyên các version** của một publication:

- **Reference entries:** các khóa khác nhau cùng trỏ tới một tài liệu tham khảo phải được gộp về **một** khóa; các lệnh `\cite{}` của khóa bị bỏ phải được đổi tên; các trường của các mục trùng phải được **hợp nhất (union)** thay vì chọn một mục và bỏ phần còn lại.
- **Full-text:** phần tử có nội dung khớp chính xác giữa các version *sau cleanup* chỉ dùng **một** `element_id`.

## 5.2. Luồng dữ liệu (Data workflow)

```text
PublicationDeduplicator.deduplicate_and_save_publication(pub_dir, clean_before_match=True)
      │
      ▼  Bước 1: load_or_build_version_data
 đọc elements/pre_dedup_{elements,hierarchy}_vX.json (hoặc elements_vX/hierarchy_vX chưa dedup);
 thiếu thì gọi HierarchyBuilder.parse_version;  version rỗng → empty_versions
      │
      ▼  Bước 2: ReferenceDeduplicator  (phần 4 + khử trùng lặp)
 load_publication_references → deduplicate_bibtex
      │   gom cụm Union-Find 3 tầng: DOI ► Title chuẩn hóa (+ tác giả/năm) ► citation_key
      │   chọn canonical_key, unionize_cluster → bảng bibtex;  key_mapping {old_key → canonical_key}
      ▼
 ghi bibtex/bibtex.json, bibtex/citation_key_mapping.json, elements/bibtex.json
      │
      ▼  Bước 3: ElementDeduplicator.deduplicate_publication
 (a) rewrite_citations_in_elements: đổi tên khóa trong mọi \cite*{…} của mọi version  ← TRƯỚC khi so khớp
 (b) clean_before_match: ensure_cleaned_content = HierarchyNormalizer.normalize_content + gọn khoảng trắng
 (c) với mỗi version tăng dần: tính chữ ký (signature) → dùng lại hoặc cấp mới element_id → ánh xạ lại cạnh
      │
      ▼  Bước 4: dựng lại cây từng version bằng HierarchyBuilder.build_nested_tree với element_id dùng chung
 ghi vX/hierarchy_tree.json, elements/{hierarchy_tree,elements,hierarchy}_vX.json
      │
      ▼  Bước 5: ba bảng chuẩn + thống kê
 elements/{elements,hierarchy}.{json,csv,parquet}, bibtex/bibtex.{csv,parquet}, elements/dedup_stats.json
```

> **Thứ tự bắt buộc:** đổi tên `\cite{}` phải xảy ra *trước* khi so khớp toàn văn. Ví dụ `v1`: `"...see \cite{Ri14}."` và `v2`: `"...see \cite{riehl}."`; sau khi đổi `Ri14 → riehl`, hai câu giống hệt và dùng chung một `element_id`.

## 5.3. Reference Deduplication — `ReferenceDeduplicator`

| Hàm | Chức năng |
| :-- | :-- |
| `normalize_title_key(title)` | Chuẩn hóa tiêu đề về chuỗi `[a-z0-9]` viết thường để so khớp |
| `extract_author_surnames(authors)` | Tập họ (viết thường) của tác giả, dùng làm chốt chặn chống gộp nhầm |
| `_are_entries_same_reference(e1, e2)` | Quyết định hai mục có cùng một tài liệu hay không (3 tầng bên dưới) |
| `deduplicate_bibtex(arxiv_id, raw_entries)` | Gom cụm bằng Union-Find kết hợp bảng băm theo DOI, tiêu đề, khóa; chọn khóa đại diện; hợp nhất trường; trả về `bibtex`, `key_mapping`, `merged_details`, `stats` |
| `unionize_cluster(arxiv_id, cluster, canonical_key)` | Hợp nhất trường của một cụm |
| `rewrite_citations_in_text(text, key_mapping)` | Đổi tên khóa trong mọi lệnh `\cite*{…}` của một chuỗi và loại khóa trùng trong cùng một lệnh |
| `rewrite_citations_in_elements(version_parsed_data, key_mapping)` | Áp dụng cho mọi phần tử của mọi version, trả về thống kê và mẫu |

**Quy tắc gom cụm trong `_are_entries_same_reference`:**

1. **Tầng 1 — DOI:** cả hai có DOI chuẩn hóa hợp lệ → trùng khi và chỉ khi DOI bằng nhau.
2. **Tầng 2 — tiêu đề chuẩn hóa** (độ dài ≥ 15, không thuộc `PLACEHOLDER_NORM_TITLES` như `articletitle`, `inpreparation`): bằng nhau thì xét thêm — nếu cả hai có năm và lệch quá 2 năm → không gộp; nếu cả hai có tập họ tác giả → gộp khi có họ chung, hoặc khi năm lệch ≤ 1 và tiêu đề dài ≥ 25 ký tự, ngược lại không gộp; nếu thiếu tập họ tác giả → gộp.
3. **Tầng 3 — cùng `citation_key`:** gộp, trừ khi tiêu đề khác nhau **và** tác giả mâu thuẫn (tác giả dùng lại một khóa cho bài khác ở version sau).

**Chọn khóa đại diện:** là khóa của phần tử đầu tiên trong cụm; nếu khóa đó đã dùng cho cụm khác thì thêm hậu tố `_<version>`. `key_mapping[old] = canonical`.

**Hợp nhất trường (`unionize_cluster`):**

| Trường | Cách chọn |
| :-- | :-- |
| `entry_type` | Kiểu có điểm ưu tiên cao nhất theo `ENTRY_TYPE_PRIORITY` (`article`/`inproceedings` > `book` > ... > `misc`) |
| `title` | Tiêu đề dài nhất |
| `authors` | Danh sách có nhiều tác giả nhất (bằng nhau thì tổng độ dài ký tự lớn hơn) |
| `year`, `doi` | Giá trị khác rỗng xuất hiện sau cùng trong cụm |
| `journal` | Ưu tiên tên chính thức hơn dạng `arXiv`/`preprint`; giữa các tên chính thức chọn tên dài hơn |
| `url` | Giá trị khác rỗng sau cùng; nếu không có mà có DOI thì tạo `https://doi.org/<doi>` |

## 5.4. Đổi tên lệnh trích dẫn

`CITE_COMMAND_PATTERN` bắt 14 lệnh: `\cite`, `\citep`, `\citet`, `\citealt`, `\citealp`, `\citeauthor`, `\citeyear`, `\citepalias`, `\citetalias`, `\parencite`, `\textcite`, `\autocite`, `\nocite`, `\hyperlinkcite`, kể cả dạng có `*` và tham số tùy chọn `[...]`. Với mỗi lệnh, danh sách khóa được đổi theo `key_mapping`; nếu hai khóa khác nhau cùng về một khóa đại diện thì chỉ giữ một (`\cite{Kabashin2009, kabashin2009phase}` → `\cite{Kabashin2009}`).

## 5.5. Full-text Deduplication — `ElementDeduplicator`

| Hàm | Chức năng |
| :-- | :-- |
| `normalize_content_key(content)` | Gộp mọi khoảng trắng thành một dấu cách |
| `ensure_cleaned_content(kind, content)` | Gọi `HierarchyNormalizer.normalize_content` rồi gộp khoảng trắng (bỏ qua nút `document`) — thực hiện yêu cầu "so khớp sau cleanup" |
| `deduplicate_publication(arxiv_id, version_parsed_data, key_mapping, clean_before_match)` | Đổi tên `\cite`, (tùy chọn) cleanup, khử trùng lặp, ánh xạ lại cạnh; trả về `elements`, `hierarchy`, `per_version_elements`, `per_version_hierarchy`, `cite_rewrite_stats`, `stats` |

**Khóa chữ ký (signature) theo loại nút:**

| Loại nút | Chữ ký | Mục đích |
| :-- | :-- | :-- |
| `document` | không khử trùng lặp: mỗi version có `<arxiv_id>#root<v>` riêng | Giữ nút gốc từng version |
| `itemize` | `(kind, nội dung các `item` con trực tiếp nối bằng " \|\| ", thứ tự xuất hiện trong version)` | Không gộp nhầm các danh sách khác nhau trong cùng version, nhưng gộp được danh sách giữ nguyên qua các version |
| tiêu đề (`part` … `subparagraph`) | `(kind, nội dung, thứ tự xuất hiện trong version)` | Hai mục trùng tên trong cùng version không bị chập nhánh |
| nút lá (`sentence`, `block_formula`, `figure`, `item`) | `(kind, nội dung chuẩn hóa)` | Khớp chính xác xuyên các version |

Chữ ký đã có thì dùng lại `element_id`; chưa có thì cấp ID mới `<arxiv_id>#<tiền tố><số thứ tự>` với tiền tố theo bảng ở mục 2.4 (cột "ID cuối cùng"). Mỗi cạnh `(version, element_id, parent_id)` được ánh xạ sang ID cuối cùng, bỏ cạnh lặp và cạnh có `parent` rỗng.

## 5.6. Điều phối — `PublicationDeduplicator`

| Hàm | Chức năng |
| :-- | :-- |
| `load_or_build_version_data(pub_dir)` | Nạp dữ liệu từng version; **idempotent** nhờ `pre_dedup_*` (bản trước dedup được lưu lại, chạy lại dedup không làm ID bị chồng thêm); thiếu thì dựng bằng `HierarchyBuilder`; ghi nhận version rỗng |
| `deduplicate_and_save_publication(pub_dir, clean_before_match=True)` | Thực hiện bước 1–5 ở sơ đồ 5.2 và trả về thống kê |

## 5.7. Đầu vào và đầu ra

| | Mô tả |
| :-- | :-- |
| **Đầu vào** | `<pub_dir>/elements/` (`pre_dedup_*`, `elements_vX`, `hierarchy_vX`, `raw_references.json`, `compilation_manifest.json`, `full_inlined_vX.tex`), `<pub_dir>/vX/*.bib`, `<pub_dir>/bibtex/` (nếu có) |
| **Bảng `bibtex`** | `bibtex/bibtex.json`, `bibtex.csv`, `bibtex.parquet`; bản sao trong `elements/`; `bibtex/citation_key_mapping.json` |
| **Bảng `elements`, `hierarchy`** | `elements/elements.{json,csv,parquet}`, `elements/hierarchy.{json,csv,parquet}` |
| **Theo version** | `vX/hierarchy_tree.json` và `elements/{hierarchy_tree,elements,hierarchy}_vX.json` với `element_id` dùng chung |
| **Thống kê** | `elements/dedup_stats.json`: số version (tổng/có TeX/rỗng/lỗi), `elements_dedup` (`total_before`, `total_after`, `reduction_percentage`, `by_kind_before/after`, số `\cite` đã đổi), `bibtex_dedup` (`total_raw_entries`, `dedup_entries_count`, `renamed_keys_count`, `fields_enriched_count`, ...), `merged_bibtex_details`, `cite_rewrite_samples` |

## 5.8. Mã nguồn chính

In [ ]:
show_source(ReferenceDeduplicator._are_entries_same_reference,
            ReferenceDeduplicator.deduplicate_bibtex,
            ReferenceDeduplicator.unionize_cluster,
            ReferenceDeduplicator.rewrite_citations_in_text)

In [ ]:
show_source(ElementDeduplicator.ensure_cleaned_content,
            ElementDeduplicator.deduplicate_publication)

In [ ]:
show_source(PublicationDeduplicator.load_or_build_version_data,
            PublicationDeduplicator.deduplicate_and_save_publication)

## 5.9. Minh họa trực tiếp

**(a) Khử trùng lặp tham khảo, đổi tên `\cite` và khử trùng lặp toàn văn** — một tài liệu được đổi khóa từ `vaswani2017` (v1, thiếu `doi`) sang `vit_transformer` (v2, thêm `doi`, `journal`, thiếu `year`); câu văn giống nhau chỉ khác khóa trích dẫn.

In [ ]:
AID = "2310.12345"
raw_bib = [
    {"arxiv_id": AID, "version": "v1", "citation_key": "vaswani2017", "entry_type": "misc",
     "title": "Attention Is All You Need", "authors": ["Vaswani, Ashish", "Shazeer, Noam"],
     "year": 2017, "doi": "", "journal": "", "url": ""},
    {"arxiv_id": AID, "version": "v2", "citation_key": "vit_transformer", "entry_type": "inproceedings",
     "title": "Attention is all you need", "authors": ["Ashish Vaswani", "Noam Shazeer", "Niki Parmar"],
     "year": None, "doi": "10.5555/3295222.3295349",
     "journal": "Advances in Neural Information Processing Systems (NeurIPS)", "url": ""},
    {"arxiv_id": AID, "version": "v2", "citation_key": "other2020", "entry_type": "misc",
     "title": "A completely different paper about graph neural networks", "authors": ["Kipf, Thomas"],
     "year": 2020, "doi": "", "journal": "", "url": ""},
]
bib_res = ReferenceDeduplicator.deduplicate_bibtex(AID, raw_bib)
print("key_mapping:", bib_res["key_mapping"])
print("stats      :", bib_res["stats"])
display(pd.DataFrame(bib_res["bibtex"]))

print("rewrite_citations_in_text:",
      ReferenceDeduplicator.rewrite_citations_in_text(r"See \citep[e.g.,][p.~5]{vaswani2017, vit_transformer} and \cite{other2020}.",
                                                      bib_res["key_mapping"]))

version_data = {
    "v1": ([{"raw_id": "root1", "kind": "document", "content": "Version 1 root"},
            {"raw_id": "h1", "kind": "section", "content": "Introduction"},
            {"raw_id": "s1", "kind": "sentence", "content": r"We adopt the Transformer \cite{vaswani2017} for matching."}],
           [{"arxiv_id": AID, "version": 1, "element_raw_id": "h1", "parent_raw_id": "root1"},
            {"arxiv_id": AID, "version": 1, "element_raw_id": "s1", "parent_raw_id": "h1"}]),
    "v2": ([{"raw_id": "root2", "kind": "document", "content": "Version 2 root"},
            {"raw_id": "h1", "kind": "section", "content": "Background and Model"},
            {"raw_id": "s1", "kind": "sentence", "content": r"We adopt the  Transformer \cite{vit_transformer}  for matching."}],
           [{"arxiv_id": AID, "version": 2, "element_raw_id": "h1", "parent_raw_id": "root2"},
            {"arxiv_id": AID, "version": 2, "element_raw_id": "s1", "parent_raw_id": "h1"}]),
}
elem_res = ElementDeduplicator.deduplicate_publication(AID, version_data, key_mapping=bib_res["key_mapping"], clean_before_match=True)
print("\nBẢNG elements (câu s1 của hai version dùng chung một dòng):")
display(pd.DataFrame(elem_res["elements"]))
print("BẢNG hierarchy (s1 gắn vào mục khác nhau ở mỗi version):")
display(pd.DataFrame(elem_res["hierarchy"]))
print("Thống kê:", {k: v for k, v in elem_res["stats"].items() if not k.startswith("by_kind")})

**(b) Chạy các bước của pipeline trên một publication thật gồm hai version.** Thư mục tạm chứa `v1` (tham khảo bằng `thebibliography`) và `v2` (đổi khóa trích dẫn, tham khảo bằng file `.bib`, thêm một câu mới). Ô này gọi `build_publication` → `normalize_publication` → `deduplicate_publication` của `ParsingPipeline` (không gọi bước dọn file TeX và benchmark) rồi liệt kê file sinh ra và ba bảng kết quả.

In [ ]:
V1_TEX = r"""\documentclass{article}
\begin{document}
\section{Introduction}
We adopt the Transformer \cite{vaswani2017} for citation matching. The model reaches 93.5 percent accuracy.
\begin{equation}
 E = m c^2
\end{equation}
\begin{thebibliography}{9}
\bibitem{vaswani2017} A. Vaswani, N. Shazeer. ``Attention is all you need.'' NeurIPS, 2017.
\end{thebibliography}
\end{document}
"""
V2_TEX = r"""\documentclass{article}
\begin{document}
\section{Introduction}
We adopt the Transformer \cite{vit_transformer} for citation matching. The model reaches 93.5 percent accuracy.
\begin{equation}
 E = m c^2
\end{equation}
This version adds one new sentence.
\bibliography{refs}
\end{document}
"""
V2_BIB = r"""@inproceedings{vit_transformer, title={{Attention Is All You Need}}, author={Vaswani, Ashish and Shazeer, Noam},
 booktitle={NeurIPS}, year={2017}, doi={10.5555/3295222.3295349}}"""

with tempfile.TemporaryDirectory() as tmp:
    src = Path(tmp) / "sourcesDemo"
    pub = src / "2310.12345"
    (pub / "v1").mkdir(parents=True); (pub / "v2").mkdir(parents=True)
    (pub / "v1" / "main.tex").write_text(V1_TEX, encoding="utf-8")
    (pub / "v2" / "main.tex").write_text(V2_TEX, encoding="utf-8")
    (pub / "v2" / "refs.bib").write_text(V2_BIB, encoding="utf-8")
    (pub / "v3").mkdir()                       # version không có TeX -> bản ghi rỗng
    (pub / "v3" / "metadata.json").write_text("{}", encoding="utf-8")

    pipe = ParsingPipeline(src, clean_redundant_tex=False)
    build = pipe.build_publication(pub)
    print("build_publication:", {k: build[k] for k in ("total_versions", "successful_versions", "empty_versions", "failed_versions")})
    print("normalize_publication ->", len(pipe.normalize_publication(pub)), "file cây đã chuẩn hóa tại chỗ")
    dd = pipe.deduplicate_publication(pub)
    print("deduplicate_publication ->", {k: dd[k] for k in ("elements_count", "hierarchy_count", "bibtex_count")})

    print("\nFile sinh ra:")
    for p in sorted(pub.rglob("*")):
        if p.is_file() and p.suffix in (".json", ".csv", ".parquet", ".tex"):
            print("  ", p.relative_to(pub))

    print("\nCompilation manifest:")
    print(json.dumps(json.loads((pub / "elements" / "compilation_manifest.json").read_text()), indent=1)[:900])
    print("\nBẢNG elements:");  display(pd.read_json(pub / "elements" / "elements.json"))
    print("BẢNG hierarchy:");   display(pd.read_json(pub / "elements" / "hierarchy.json"))
    print("BẢNG bibtex:");      display(pd.read_json(pub / "bibtex" / "bibtex.json"))
    print("citation_key_mapping:", json.loads((pub / "bibtex" / "citation_key_mapping.json").read_text()))
    stats = json.loads((pub / "elements" / "dedup_stats.json").read_text())
    print("elements_dedup:", {k: v for k, v in stats["elements_dedup"].items() if not k.startswith("by_kind")})
    print("bibtex_dedup  :", stats["bibtex_dedup"])
    print("Version rỗng v3 -> root của cây:", json.loads((pub / "v3" / "hierarchy_tree.json").read_text())["root"])

---

# 6. Benchmark và thống kê (`benchmark.py`)

## 6.1 Mục đích và data workflow

`ParserBenchmark` thu thập số liệu **trong lúc pipeline chạy** để đánh giá hiệu năng, độ phủ và hiệu quả khử trùng lặp. Lớp này không tham gia xử lý dữ liệu LaTeX; `ParsingPipeline` gọi nó ở các điểm cố định.

```text
run_all / main ── benchmark.start()                    (bật tracemalloc, ghi mốc thời gian, RSS ban đầu)
      │
      ├─ mỗi version  ──► record_version(arxiv_id, version, status, duration, error_msg,
      │                                  compilation_files_count, elements_count)
      ├─ mỗi publication ► record_publication(expected, crawled, tex, empty, dedup_stats, duration)
      │
      └─ benchmark.stop()                              (lấy đỉnh tracemalloc, tắt tracemalloc, cập nhật RSS)
                │
                ▼
      generate_summary() ──► dict 4 nhóm: performance · success_rate · version_coverage · deduplication
      export_markdown_report(path) ──► file Markdown (CLI: --report-md; JSON: --stats-json)
```

Trạng thái mỗi version là một trong ba giá trị: `success`, `empty_version` (không có file `.tex`), `failed` (mọi lỗi khác).

## 6.2 Lớp và hàm chính

| Thành phần | Chức năng | Input → Output |
| :-- | :-- | :-- |
| `ParserBenchmark` | Lưu `version_logs`, `pub_stats` và các mốc thời gian/bộ nhớ | — |
| `start()` / `stop()` | Bắt đầu / kết thúc đo: `tracemalloc`, `time.perf_counter`, RSS | — → — |
| `update_peak_memory()` | Lấy RSS hiện tại (`psutil`), giữ giá trị lớn nhất | — → — |
| `record_version(...)` | Ghi kết quả một version | thông tin version → thêm vào `version_logs` |
| `record_publication(...)` | Ghi thống kê một publication, lấy `total_before/after`, `reduction_*`, `by_kind_*` từ `dedup_stats` | thông tin publication → thêm vào `pub_stats` |
| `generate_summary()` | Tổng hợp toàn cục | — → `dict` |
| `export_markdown_report(path)` | Ghi báo cáo Markdown (3 mục: hiệu năng, tỉ lệ thành công và coverage, khử trùng lặp theo `kind`) | đường dẫn → chuỗi Markdown |

## 6.3 Công thức các chỉ số

| Chỉ số | Công thức trong mã |
| :-- | :-- |
| Tỉ lệ thành công | `successful_versions / total_versions_processed × 100` |
| Version coverage | `versions_with_tex / total_expected_versions × 100` (tổng theo các publication) |
| Tỉ lệ giảm trùng lặp | `(total_before − total_after) / total_before × 100`, cũng tính riêng cho từng `kind` |
| Thời gian TB mỗi publication / version | `total_wall_time / số publication` và `total_wall_time / số version` |
| `peak_rss_mb` | RSS lớn nhất đã lấy mẫu `/ 1024²` |
| `peak_tracemalloc_mb` | đỉnh heap Python do `tracemalloc` ghi nhận `/ 1024²` |

**Lưu ý khi đọc số liệu (đối chiếu với mã nguồn):**

1. RSS **chỉ được lấy mẫu** tại `start`, mỗi lần `record_version`, `record_publication` và `stop`, nên `peak_rss_mb` là đỉnh trong các thời điểm lấy mẫu, không phải đỉnh thật liên tục.
2. `tracemalloc` chỉ theo dõi cấp phát của Python và làm chậm chương trình, nên thời gian khi bật benchmark cao hơn lúc không đo.
3. Thời gian trung bình lấy **tổng thời gian chạy chia số lượng**, nên bao gồm cả thời gian dọn file và ghi file của pipeline, không chỉ thời gian parse.
4. Trong pipeline, `expected_versions` và `crawled_versions` đều lấy bằng tổng số version tìm thấy trên đĩa, vì vậy coverage đo tỉ lệ version **có TeX** trên các version có thư mục, không phản ánh version bị thiếu từ lúc crawl.

In [ ]:
# Mã nguồn các hàm chính của ParserBenchmark
show_source(ParserBenchmark.start, ParserBenchmark.update_peak_memory, ParserBenchmark.stop,
            ParserBenchmark.record_version, ParserBenchmark.record_publication)

In [ ]:
show_source(ParserBenchmark.generate_summary, ParserBenchmark.export_markdown_report)

## 6.4 Demo: chạy pipeline có benchmark trên dữ liệu mô phỏng

Ô dưới tạo thư mục tạm gồm hai publication (một publication có hai version kèm một version rỗng; publication kia có một version), chạy `ParsingPipeline.run_all()` (đã gồm `benchmark.start()` / `stop()`), rồi in các nhóm số liệu của `generate_summary()`. **Đây là dữ liệu mô phỏng nhỏ để minh họa cơ chế, không phải kết quả của lần chạy thật trên toàn bộ tập publication.**

In [ ]:
TEX_A1 = r"""\documentclass{article}
\begin{document}
\section{Introduction}
We study citation matching. The model is simple.
\begin{thebibliography}{9}
\bibitem{k1} A. Vaswani, N. Shazeer. ``Attention is all you need.'' NeurIPS, 2017.
\end{thebibliography}
\end{document}
"""
TEX_A2 = TEX_A1.replace("The model is simple.", "The model is simple. We add one more sentence.")
TEX_B1 = r"""\documentclass{article}
\begin{document}
\section{Method}
A single version publication. It has two sentences.
\end{document}
"""
with tempfile.TemporaryDirectory() as tmp:
    src = Path(tmp) / "sourcesBench"
    layout = {"2310.00001": {"v1": TEX_A1, "v2": TEX_A2, "v3": None}, "2310.00002": {"v1": TEX_B1}}
    for aid, vers in layout.items():
        for v, tex in vers.items():
            (src / aid / v).mkdir(parents=True)
            if tex is not None:
                (src / aid / v / "main.tex").write_text(tex, encoding="utf-8")
    pipe = ParsingPipeline(src, clean_redundant_tex=False)
    summary = pipe.run_all()
    report_md = pipe.benchmark.export_markdown_report(Path(tmp) / "benchmark_report.md")

    for group, vals in summary.items():
        print(f"--- {group} ---")
        display(pd.DataFrame([{k: v for k, v in vals.items() if not isinstance(v, dict)}]))
    kinds = summary["deduplication"]
    df_kind = pd.DataFrame({"trước dedup": kinds["by_kind_before"], "sau dedup": kinds["by_kind_after"]}).fillna(0).astype(int)
    df_kind["giảm"] = df_kind["trước dedup"] - df_kind["sau dedup"]
    print("--- Phân bố theo kind ---"); display(df_kind)
    print("--- Nhật ký từng version ---")
    display(pd.DataFrame(pipe.benchmark.version_logs)[["arxiv_id", "version", "status", "duration_sec", "compilation_files_count", "elements_count"]])
    print("--- Báo cáo Markdown (đầu file) ---"); print("\n".join(report_md.splitlines()[:12]))

**Cách đọc kết quả demo:** `empty_version` (v3) được tính vào `total_versions_processed` nên kéo `success_rate` xuống dưới 100% nhưng không phải lỗi; `coverage` tính theo version có TeX; các câu và mục giống nhau giữa v1 và v2 được gộp nên `reduction_percentage` dương. Khi chạy thật bằng CLI, cùng số liệu được ghi vào `benchmark_report.md` và `benchmark_stats.json`.

---

# Tổng kết: đầu vào – đầu ra theo từng phần

| Phần | Đầu vào | Đầu ra |
| :-- | :-- | :-- |
| 1. Multi File Gathering | `vX/*.tex` (+ `metadata.json` tùy chọn) | `full_inlined_vX.tex`, `compilation_manifest.json` (`root_file`, `compilation_files`, `orphan_files`, trạng thái `success` / `empty_version` / `failed`) |
| 2. Hierarchy Construction | Chuỗi LaTeX đã gom | `elements_vX`, `hierarchy_vX`, `hierarchy_tree_vX` / `vX/hierarchy_tree.json` (nội dung thô), `pre_dedup_*_vX.json`, `raw_references.json` |
| 3. Cleanup and Normalization | Các file `hierarchy_tree*.json` | Cùng cây với `content` đã chuẩn hóa toán và làm sạch LaTeX (ghi tại chỗ trong pipeline; hoặc `*.normalized.json` khi chạy CLI) |
| 4. Reference Extraction | `.bib` trong `vX/`, `raw_references.json` (`\bibitem`), `bibtex/` (nếu có) | Danh sách mục BibTeX thô theo version (10 trường chuẩn) |
| 5. Deduplication | Phần tử, cạnh theo version; các mục BibTeX thô | `elements`, `hierarchy`, `bibtex` (JSON/CSV/Parquet), `citation_key_mapping.json`, cây từng version với ID dùng chung, `dedup_stats.json` |

## Các điểm cần lưu ý khi đọc kết quả (đã đối chiếu với mã nguồn)

1. **`align`, `gather`, `multline`, `eqnarray`, `flalign` được đổi thành `equation`:** thân công thức được giữ nguyên nhưng ký hiệu căn lề `&` và ngắt dòng `\\` không còn đúng ngữ nghĩa của `equation`, và dạng `*` (không đánh số) cũng thành dạng đánh số; kết quả phù hợp để so khớp văn bản, không nên biên dịch lại như LaTeX gốc.
2. **Môi trường toán khối lồng nhau hoặc ghép cặp không chắc chắn được giữ nguyên;** một thẻ mở không có thẻ đóng khiến các môi trường đứng sau nó trong cùng nút cũng giữ nguyên.
3. **Bước chuẩn hóa chỉ đổi môi trường toán cho nút `kind = "block_formula"`;** các nút khác (ví dụ `itemize`, `figure`) chỉ được `standardize_math` và `clean_latex` xử lý.
4. **Sau Deduplication, cây từng version được dựng lại** từ các phần tử đã qua `ensure_cleaned_content` (vì `clean_before_match = True` là mặc định), nên `content` trong bảng `elements` đã được chuẩn hóa và gộp khoảng trắng.
5. **Thư mục `elements/` cũng chứa `hierarchy_tree_vX.json`,** nên pattern `hierarchy_tree*.json` của `normalize_path` khớp cả file trong `vX/` lẫn trong `elements/` khi quét một thư mục publication.
6. **Phát hiện chu trình và nút gốc rỗng:** `build_nested_tree` báo lỗi nếu thiếu nút gốc, cạnh trỏ tới nút không tồn tại hoặc có chu trình; `pipeline` bắt lỗi khi `parse_version` thất bại và ghi nhận version là `failed`.
7. **Sửa so với các báo cáo thành phần:** (a) `gatherer.py` chỉ trả kết quả trong bộ nhớ và không import `json`/`shutil`; `compilation_manifest.json` và `full_inlined_vX.tex` do `pipeline.py` ghi; (b) `elements.json`/`hierarchy.json` do phần Deduplication ghi, không phải `hierarchy_builder.py`; (c) normalizer hiện tự chứa `standardize_math`/`clean_latex`, không còn phụ thuộc `HierarchyExtractor`; (d) môi trường `appendix` được đổi thành `\appendix` (khi bên trong đã có `\section`/`\chapter`) hoặc `\section*{Appendix}`, và `abstract` cũng được đổi thành `\section*{Abstract}`; (e) ID cuối cùng của phần tử do phần Deduplication cấp (`#s1`, `#sec1`, ...), khác ID trung gian `#v1_s1`, `#v1_h1`.
8. **Các lỗi đã được sửa trong phiên bản code hiện tại** (không còn là hạn chế): (a) `parse_authors_list` tách được danh sách tác giả `\bibitem` phân cách bằng dấu phẩy, nên tầng 2 so khớp họ tác giả được cho cả mục `\bibitem` lẫn `.bib`; (b) khóa `"booklet"` của `ENTRY_TYPE_PRIORITY` không còn dấu cách thừa; (c) khi chạy lại mà `full_inlined_vX.tex` được dùng lại, manifest không còn bị ghi đè bằng danh sách file rỗng; (d) trong nhánh tách theo dấu chấm của `parse_bibitem_entries`, năm đứng cuối phần nơi xuất bản bị bỏ khi trùng với `year`.